# G31-04 Stock-Out Risk Prediction: data preparation (Progress 1)
Group G31-04, Dataset S1 (e-commerce events and item properties, 3 May to 18 Sep 2015).
Data Engineer part: cleaning, weekly item table with the target, category fixes, demand features, data dictionary,
cleaning log, KPI tables and all charts.

**This notebook is standalone.** All code is inside. It needs only the 4 raw CSV files of Dataset S1:
`item_properties_part1.csv`, `item_properties_part2.csv`, `events.csv`, `category_tree.csv`.

**How to run (Google Colab or Jupyter)**
1. In cell 1, choose how to give the 4 raw files:
   - `download` (default): downloads them from the group's public Google Drive folder. No sign-in needed.
   - `upload`: Colab shows a button to pick the files (or one zip) from your computer.
   - `folder`: the files are already in a folder (type its path in `RAW_FOLDER`). In Colab you can also drag the 4 files
     into the folder `G31-04_run/data_raw` in the *Files* panel on the left; they are then used directly.
2. *Runtime > Run all* (Colab) or *Run > Run All Cells* (Jupyter). A full run takes about 5 to 10 minutes and about 4 GB RAM.
3. All results are in the folder `G31-04_run` and in `G31-04_Progress1_results.zip` (downloaded at the end in Colab).

Every number below is computed by this code. Nothing is typed by hand, except the numbers from the proposal and the
Progress 1 slides that the results are compared with (they are marked as such).

In [ ]:
#@title 0. Setup: folders and libraries
import os, sys, glob, shutil, zipfile, subprocess, importlib.util, gc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUN_DIR = Path(os.environ.get("NB_RUN_DIR", "/content/G31-04_run" if IN_COLAB else "G31-04_run")).resolve()
DATA_DIR, WORK_DIR, OUT_DIR, DOCS_DIR = RUN_DIR / "data_raw", RUN_DIR / "work", RUN_DIR / "outputs", RUN_DIR / "docs"
for d in (DATA_DIR, WORK_DIR, OUT_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
UNKNOWN_CAT = -1   # code for the "Unknown" category (no categoryid known at or before the snapshot)

missing = [m for m in ["pandas", "numpy", "openpyxl", "matplotlib", "gdown"] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import pandas, numpy, matplotlib, openpyxl
print("run folder:", RUN_DIR)
print("python", sys.version.split()[0], "| pandas", pandas.__version__, "| numpy", numpy.__version__,
      "| matplotlib", matplotlib.__version__, "| openpyxl", openpyxl.__version__)


def _free(before):
    """Forget the variables a step made (as if every step were a new program) and free the memory."""
    for k in list(globals()):
        if k not in before and not k.startswith("_") and k not in ("In", "Out"):
            del globals()[k]
    gc.collect()

In [ ]:
#@title 0b. Cleaning log helper (rows and columns before and after each step)
import pandas as pd

PATH = WORK_DIR / "cleaning_log.csv"
COLS = ["script", "step", "action", "table", "rows_before", "cols_before", "rows_after", "cols_after",
        "rows_changed", "note"]


class CleanLog:
    def __init__(self, script):
        self.script = script
        self.n = 0
        old = pd.read_csv(PATH) if PATH.exists() else pd.DataFrame(columns=COLS)
        self.rows = old[old["script"] != script].to_dict("records")
        self._save()

    def add(self, action, table, before, after, note=""):
        """before and after are (rows, cols) tuples."""
        self.n += 1
        rb, cb = before
        ra, ca = after
        self.rows.append({"script": self.script, "step": self.n, "action": action, "table": table,
                          "rows_before": int(rb), "cols_before": int(cb), "rows_after": int(ra), "cols_after": int(ca),
                          "rows_changed": int(ra) - int(rb), "note": note})
        print(f"[log] {action}: {table} {rb:,} x {cb} -> {ra:,} x {ca}. {note}", flush=True)
        self._save()

    def _save(self):
        df = pd.DataFrame(self.rows, columns=COLS)
        df["order"] = df["script"].str[:2]
        df.sort_values(["order", "step"], kind="stable").drop(columns="order").to_csv(PATH, index=False)

LOG_PATH = PATH

In [ ]:
#@title 0c. Data dictionary text: type, meaning, used, why (example values come from the data in step 5)
# (column, type, meaning, used, why)
SPEC = {
    "item_properties (part1 + part2)": [
        ("timestamp", "integer (Unix ms, UTC)", "Time when the property value was recorded.", "yes",
         "Puts status and category in time order and gives the 18 weekly snapshot dates."),
        ("itemid", "integer", "Anonymous item id.", "yes", "Key to join properties with events."),
        ("property", "text", "Property name. Most names are hashed numbers; `available` and `categoryid` are plain.", "yes",
         "Used to keep only `available` and `categoryid` (step 1). The hashed ones have no known meaning."),
        ("value", "text", "Property value. For `available` it is 0 or 1, for `categoryid` a category id, else hashed text.",
         "yes", "Gives the stock status (label and eligibility) and the item category."),
    ],
    "events": [
        ("timestamp", "integer (Unix ms, UTC)", "Time of the event.", "yes",
         "Puts each event in the 7-day and 28-day windows before a snapshot."),
        ("visitorid", "integer", "Anonymous visitor id.", "yes (step 4)",
         "Counts distinct visitors per item (`visitors_1w`)."),
        ("event", "text", "Event type: view, addtocart or transaction.", "yes",
         "Demand counts by type and the active flag."),
        ("itemid", "integer", "Anonymous item id.", "yes", "Key to join events with item status."),
        ("transactionid", "number, empty unless event = transaction", "Id of the purchase.", "no",
         "Purchases are already counted from the event rows. The id adds nothing for this task."),
    ],
    "category_tree": [
        ("categoryid", "integer", "Category id (same codes as `value` of property `categoryid`).", "yes (step 3)",
         "Key to find the parent of each item category (`parent_cat`)."),
        ("parentid", "number, empty for top-level categories", "Id of the parent category.", "yes (step 3)",
         "Gives `parent_cat`: groups similar categories, so small categories still share information. Can later be used for KPI 2 at parent level."),
    ],
    "modelling table (work/modelling_table_features.csv.gz), one row per item and prediction week": [
        ("itemid", "integer", "Item id.", "key", "Identifies the row together with `snapshot`."),
        ("snapshot", "text date YYYY-MM-DD", "Snapshot date t of the row (16 prediction weeks).", "key",
         "Time key. Used for the train / validate / test split by date."),
        ("state", "0/1", "Stock status at t after forward fill. Always 1 in this table.", "check only",
         "Kept to show the eligibility filter (in stock at t). Not a feature."),
        ("cat", "integer", "Last known categoryid at or before t. -1 = Unknown (no category known yet).", "feature",
         "Stock-out risk differs by category, and KPI 2 is reported per category."),
        ("cat_missing", "0/1", "1 if `cat` is Unknown (-1).", "feature",
         "Lets a model treat missing category on its own instead of as a real code."),
        ("parent_cat", "integer", "Parent of `cat` in category_tree. A top-level category is its own parent. -1 = Unknown (cat is Unknown or not in the tree).",
         "feature", "Proposal feature. Small categories have few rows; the parent level gives the model more rows per group."),
        ("cat_in_tree", "0/1", "1 if `cat` is found in category_tree.", "feature",
         "Marks categories that are in the item data but not in the tree, so their Unknown parent is not read as a real group."),
        ("views_1w", "integer", "Views of the item in the 1w window.", "feature",
         "Short-term interest. Part of the demand definition."),
        ("views_4w", "integer", "Views in the 4w window.", "feature", "Longer, more stable interest level."),
        ("atc_1w", "integer", "Add-to-carts in the 1w window.", "feature",
         "Stronger buying intent than a view. Part of the demand definition."),
        ("atc_4w", "integer", "Add-to-carts in the 4w window.", "feature", "Longer-run intent level."),
        ("tx_1w", "integer", "Purchases (transaction events) in the 1w window.", "feature",
         "Purchases are what actually take units out of stock."),
        ("tx_4w", "integer", "Purchases in the 4w window.", "feature", "Recent sales level, less noisy than one week."),
        ("demand_1w", "integer", "views_1w + atc_1w (proposal definition of demand).", "feature",
         "Main demand measure. Also the weight in the demand-weighted KPI 2 score and in KPI 3."),
        ("active", "true/false", "At least one event in the 1w window. Always true in this table.", "check only",
         "Kept to show the eligibility filter (active at t). Not a feature."),
        ("changes_4w", "integer", "Number of status changes over the last 4 snapshot steps.", "feature",
         "Items that switched status recently tend to switch again. It is also the KPI 4 rule baseline."),
        ("y_next_out", "0/1", "Label: 1 if the item is out of stock at the snapshot 7 days after t.", "label",
         "Target of the classification model."),
        ("demand_4w", "integer", "views_4w + atc_4w.", "feature", "Demand level over a month, smoother than demand_1w."),
        ("demand_prev3w_avg", "number", "(demand_4w - demand_1w) / 3, the weekly average of the 3 weeks before the last week.",
         "feature", "Baseline to compare the last week against."),
        ("demand_trend", "number", "(demand_1w + 1) / (demand_prev3w_avg + 1). Above 1 = demand is rising.", "feature",
         "A sudden rise in demand can empty stock faster than it is refilled. The +1 avoids division by zero."),
        ("atc_rate_1w", "number", "atc_1w / views_1w, 0 if no views.", "feature",
         "Share of views that turn into intent. High intent with the same views means more units are likely to be sold. "
         "Can be above 1 when an item is added to cart without a view in the window."),
        ("buy_rate_4w", "number", "tx_4w / views_4w, 0 if no views.", "feature",
         "How well views turn into sales over a month. Can be above 1 in rare cases (purchases without a view in the window)."),
        ("visitors_1w", "integer", "Distinct visitors with a view or add-to-cart in the 1w window.", "feature",
         "Separates broad demand (many people) from one visitor who views the same item many times."),
        ("days_since_last_tx", "number, can be empty", "Days from the last purchase at or before t to t. Empty = no purchase seen since 3 May.",
         "feature", "Items that sold recently are more likely to run out. Fill the empty values at model time (for example with a large number plus a flag)."),
        ("cat_demand_1w", "integer, empty for Unknown", "Sum of demand_1w of all active items in the same category at t (in stock or not).",
         "feature", "How busy the whole category is. Out-of-stock siblings can push demand to the in-stock item."),
        ("item_share_of_cat_demand", "number, empty for Unknown", "demand_1w / cat_demand_1w.", "feature",
         "Whether the item is a main seller in its category or a small one."),
    ],
}

## 1. The raw data

In [ ]:
#@title 1. Get the 4 raw CSV files
RAW_SOURCE = "download"  #@param ["download", "upload", "folder"]
DOWNLOAD_LINK = "https://drive.google.com/drive/folders/1L2N2pFDTrI8cf2VQQMiS9PuqRIN0EuIq"  #@param {type:"string"}
RAW_FOLDER = ""  #@param {type:"string"}
RAW_SOURCE = os.environ.get("NB_RAW_SOURCE", RAW_SOURCE)     # only for testing outside Colab
RAW_FOLDER = os.environ.get("NB_RAW_FOLDER", RAW_FOLDER)

NEEDED = ["item_properties_part1.csv", "item_properties_part2.csv", "events.csv", "category_tree.csv"]
have = lambda: [n for n in NEEDED if (DATA_DIR / n).exists()]


def collect(src):
    """Copy the 4 CSVs from a folder (any sub-folder) into data_raw/, or extract them from a zip there."""
    for name in NEEDED:
        if (DATA_DIR / name).exists():
            continue
        hits = glob.glob(f"{src}/**/{name}", recursive=True)
        if hits:
            shutil.copy(hits[0], DATA_DIR / name)
            continue
        for z in glob.glob(f"{src}/**/*.zip", recursive=True):
            with zipfile.ZipFile(z) as zf:
                member = next((m for m in zf.namelist() if m.endswith(name)), None)
                if member:
                    with zf.open(member) as s, open(DATA_DIR / name, "wb") as d:
                        shutil.copyfileobj(s, d)
                    break


if len(have()) < len(NEEDED):
    if RAW_SOURCE == "download":
        import gdown
        tmp = RUN_DIR / "download"
        if "/folders/" in DOWNLOAD_LINK:
            gdown.download_folder(url=DOWNLOAD_LINK, output=str(tmp), quiet=False, use_cookies=False)
        else:   # a link to one file, for example the course zip
            tmp.mkdir(exist_ok=True)
            gdown.download(url=DOWNLOAD_LINK, output=str(tmp / "raw_download.zip"), quiet=False, fuzzy=True)
        collect(tmp)
    elif RAW_SOURCE == "upload":
        if not IN_COLAB:
            raise RuntimeError(f"Upload works in Colab only. Copy the 4 files into {DATA_DIR} and run this cell again.")
        from google.colab import files
        up = RUN_DIR / "upload"
        up.mkdir(exist_ok=True)
        for name, data in files.upload().items():   # pick the 4 CSV files, or one zip with them
            (up / name).write_bytes(data)
        collect(up)
    else:
        collect(RAW_FOLDER)

miss = [n for n in NEEDED if n not in have()]
if miss:
    raise FileNotFoundError(f"Missing raw files: {miss}. Check RAW_SOURCE, DOWNLOAD_LINK or RAW_FOLDER in this cell.")
for n in NEEDED:
    print(f"{n:28s} {(DATA_DIR / n).stat().st_size:>13,} bytes")

In [ ]:
#@title 1b. A first look: column names and the first 3 rows of each raw file
from IPython.display import display
for n in NEEDED:
    print(n)
    display(pandas.read_csv(DATA_DIR / n, nrows=3))

## 2. The pipeline (steps 1 to 7)
Each step reads what the step before saved in `G31-04_run/`, so the steps can be checked one by one.

### Step 1: read the raw files and first cleaning
Keeps only the 2 readable item properties (`available`, `categoryid`) out of all properties, and removes exact duplicate events.

```text
Step 1: read the raw files and do the first cleaning.

a) item_properties part 1 and 2: keep only property in {available, categoryid}. Read in chunks so RAM stays low.
   Output: work/item_props_small.csv.gz
b) events.csv: remove exact duplicate rows (same values in all 5 columns). Output: work/events_clean.csv.gz
   All later steps read the cleaned events, never the raw file.
Step 1 also starts a new cleaning log (work/cleaning_log.csv), so every full run gives one clean log.
```
*Code copied from `scripts/01_extract_raw.py`.*

In [ ]:
#@title Step 1: read the raw files and first cleaning (part 1 of 3)
_before = set(globals())
import pandas as pd

LOG_PATH.unlink(missing_ok=True)          # new run, new log
log = CleanLog("01_extract_raw")

In [ ]:
#@title Step 1: read the raw files and first cleaning (part 2 of 3)
# ---------- a) item properties ----------
KEEP = {"available", "categoryid"}
parts, n_raw, n_cols, props = [], 0, 0, set()
for name in ("item_properties_part1.csv", "item_properties_part2.csv"):
    total = 0
    for ch in pd.read_csv(DATA_DIR / name, chunksize=2_000_000,
                          dtype={"timestamp": "int64", "itemid": "int64", "property": "string", "value": "string"}):
        total += len(ch)
        n_cols = ch.shape[1]
        props.update(ch["property"].unique())
        parts.append(ch[ch["property"].isin(KEEP)])
    print(name, "rows read:", total, flush=True)
    log.add("read raw file", name, (0, 0), (total, n_cols), "chunks of 2,000,000 rows")
    n_raw += total
df = pd.concat(parts, ignore_index=True)
log.add("stack part 1 and part 2", "item_properties (both parts)", (n_raw, n_cols), (n_raw, n_cols),
        f"long format: one row = one property value of one item at one time; {len(props):,} different properties")
log.add("keep property in {available, categoryid}", "item_properties -> item_props_small", (n_raw, n_cols), df.shape,
        f"the other {len(props) - len(KEEP):,} properties have coded names and values; "
        + ", ".join(f"{k}: {v:,}" for k, v in df["property"].value_counts().items()))
df.to_csv(WORK_DIR / "item_props_small.csv.gz", index=False)
print("kept rows:", len(df), "| properties in raw data:", len(props))

In [ ]:
#@title Step 1: read the raw files and first cleaning (part 3 of 3)
# ---------- b) events ----------
ev = pd.read_csv(DATA_DIR / "events.csv")
log.add("read raw file", "events.csv", (0, 0), ev.shape)
n0 = ev.shape
ev = ev.drop_duplicates(ignore_index=True)
log.add("drop exact duplicate events (all 5 columns equal)", "events -> events_clean", n0, ev.shape,
        f"{n0[0] - len(ev):,} duplicate rows removed (keep the first)")
ev.to_csv(WORK_DIR / "events_clean.csv.gz", index=False)
print("events:", n0[0], "->", len(ev))
_free(_before)

### Step 2: check the weekly snapshot dates
Prints the dates on which `available` was recorded, and basic counts. Nothing is saved.

```text
Step 2 (exploration): find the weekly snapshot dates of `available` and basic coverage.
```
*Code copied from `scripts/02_explore_snapshots.py`.*

In [ ]:
#@title Step 2: check the weekly snapshot dates
_before = set(globals())
import pandas as pd
p = pd.read_csv(WORK_DIR / "item_props_small.csv.gz", dtype={"value": "string"})
a = p[p.property == "available"].copy()
a["dt"] = pd.to_datetime(a["timestamp"], unit="ms")
a["day"] = a["dt"].dt.date
print("distinct timestamps:", a["timestamp"].nunique(), "| distinct days:", a["day"].nunique())
print(a.groupby("day").size().to_string())
print("values:", a["value"].value_counts().to_dict())
print("items with availability:", a["itemid"].nunique())
c = p[p.property == "categoryid"]
print("items with category:", c["itemid"].nunique(), "| categories:", c["value"].nunique())
e = pd.read_csv(WORK_DIR / "events_clean.csv.gz")   # duplicates removed in step 1
e["dt"] = pd.to_datetime(e["timestamp"], unit="ms")
print("events:", len(e), e["dt"].min(), e["dt"].max(), "| items:", e["itemid"].nunique(), "| visitors:", e["visitorid"].nunique())
print(e["event"].value_counts().to_dict())
_free(_before)

### Step 3: weekly item table, target, category fixes, KPI tables
One row per item and week, status carried forward, the target (out of stock 7 days later), the 'Unknown' and parent category, the 4 KPI tables, and the check against the numbers written in the proposal (Reconcile).

```text
Step 3: build the weekly item table and the 4 KPI tables.

Definitions (follow G31-04 Proposal, Chapter 1.2 and 2.2):
  state(item, t)  = last known value of `available` at or before snapshot t (forward fill). 1 = in stock, 0 = out.
  active(item, t) = at least one event (view, addtocart, transaction) in (t - 7 days, t].
  demand          = views + add-to-carts in (t - 7 days, t].
  next-week label = state at the snapshot exactly 7 days after t (needs that snapshot to exist).
  category        = last known categoryid at or before t. If there is none, cat = -1 ("Unknown") and cat_missing = 1.
  parent category = parentid of cat in category_tree.csv. A top-level category is its own parent. If cat is Unknown or
                    not in the tree, parent_cat = -1 ("Unknown") and cat_in_tree = 0.
                    "Unknown" is kept in the modelling table and shown in the KPI sheets, but it is not ranked in KPI 2.
Outputs: work/modelling_table.csv.gz, work/item_week.csv.gz (input for step 4) and outputs/G31-04 KPI Tables.xlsx
Row and column counts of each cleaning step go to work/cleaning_log.csv.
```
*Code copied from `scripts/03_build_kpis.py`.*

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 1 of 7)
_before = set(globals())
import numpy as np
import pandas as pd

DAY = 86_400_000
log = CleanLog("03_build_kpis")
p = pd.read_csv(WORK_DIR / "item_props_small.csv.gz", dtype={"value": "string"})
log.add("load step 1 output", "item_props_small", (0, 0), p.shape)
ev = pd.read_csv(WORK_DIR / "events_clean.csv.gz", usecols=["timestamp", "event", "itemid"])
log.add("load cleaned events (3 of 5 columns)", "events_clean", (0, 0), ev.shape, "visitorid and transactionid not needed here")

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 2 of 7)
# ---------- snapshots and state matrices (items x snapshots) ----------
a0 = p[p.property == "available"]
a = a0.drop_duplicates(["itemid", "timestamp"], keep="last")
log.add("drop duplicate (itemid, timestamp) for available", "available rows", a0.shape, a.shape, "keep the last row")
_s = a.sort_values(["itemid", "timestamp"])
n_first_or_change = int((_s["value"] != _s.groupby("itemid")["value"].shift()).sum())
log.add("count first record and each status change (check only)", "available rows", a.shape, (n_first_or_change, a.shape[1]),
        "the status is repeated every week even without a change; this count is not used further")
snaps = np.sort(a["timestamp"].unique())
dates = pd.to_datetime(snaps, unit="ms").strftime("%Y-%m-%d")
S = a.pivot(index="itemid", columns="timestamp", values="value").astype(float).reindex(columns=snaps)
na_before = int(S.isna().sum().sum())
log.add("pivot available to item x snapshot status matrix", "status matrix", a.shape, S.shape,
        f"rows = items, columns = {len(snaps)} snapshots; empty cells: {na_before:,}")
S = S.ffill(axis=1)                       # carry the last known status forward
na_after = int(S.isna().sum().sum())
log.add("forward fill status across snapshots", "status matrix", S.shape, S.shape,
        f"empty cells {na_before:,} -> {na_after:,} (left empty = before the first known status)")
items = S.index
n_known = int(S.notna().sum().sum())
log.add("item-weeks with a known status (long format: itemid, snapshot, status)", "status matrix -> item-week rows",
        (S.size, 3), (n_known, 3), f"{len(items):,} items x {len(snaps)} snapshots = {S.size:,}, minus {na_after:,} before the first status")
c0 = p[p.property == "categoryid"]
c = c0.drop_duplicates(["itemid", "timestamp"], keep="last")
log.add("drop duplicate (itemid, timestamp) for categoryid", "categoryid rows", c0.shape, c.shape, "keep the last row")
C = c.pivot(index="itemid", columns="timestamp", values="value").astype(float)
C = C.reindex(columns=sorted(set(C.columns) | set(snaps))).ffill(axis=1)[snaps].reindex(items)
log.add("category per item and snapshot (last known, forward fill)", "category matrix", c.shape, C.shape,
        f"rows = items with a status; {int(C.isna().any(axis=1).sum()):,} items have at least one snapshot without a category")
# Missing category: either the item never has a categoryid, or its first categoryid comes after the snapshot.
never = ~C.index.isin(c["itemid"].unique())
n_miss = int(C.isna().sum().sum())
n_never = int(C[never].isna().sum().sum())
C = C.fillna(UNKNOWN_CAT)
log.add('fill missing category with "Unknown" (cat = -1)', "category matrix", C.shape, C.shape,
        f"cells filled: {n_miss:,} ({n_never:,} where the item never has a categoryid, {n_miss - n_never:,} where it is only known later)")
Sv, Cv = S.to_numpy(), C.to_numpy()

# parent category from the category tree (small file, read whole)
tree = pd.read_csv(DATA_DIR / "category_tree.csv")
log.add("load category tree", "category_tree", (0, 0), tree.shape,
        f"{int(tree['parentid'].isna().sum())} top-level categories have no parent")
PARENT = {int(c): int(c if pd.isna(p) else p) for c, p in zip(tree["categoryid"], tree["parentid"])}
item_cats = set(c["value"].astype(float).astype(int))
not_in_tree = sorted(item_cats - set(PARENT))
print("items:", len(items), "snapshots:", len(snaps))

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 3 of 7)
# ---------- demand per snapshot ----------
n0 = ev.shape
ev = ev[ev["event"].isin(["view", "addtocart", "transaction"])]
log.add("keep event in {view, addtocart, transaction}", "events", n0, ev.shape)
def window(t, days):
    w = ev[(ev["timestamp"] > t - days * DAY) & (ev["timestamp"] <= t)]
    return w.groupby(["itemid", "event"]).size().unstack(fill_value=0).reindex(columns=["view", "addtocart", "transaction"], fill_value=0)

frames = {}
for j, t in enumerate(snaps):
    w1, w4 = window(t, 7), window(t, 28)
    d = pd.DataFrame(index=items)
    d["state"] = Sv[:, j]
    d["cat"] = Cv[:, j].astype("int64")
    d["cat_missing"] = (d["cat"] == UNKNOWN_CAT).astype(int)
    d["parent_cat"] = d["cat"].map(PARENT).fillna(UNKNOWN_CAT).astype("int64")
    d["cat_in_tree"] = d["cat"].isin(list(PARENT)).astype(int)
    for col, nm in (("view", "views"), ("addtocart", "atc"), ("transaction", "tx")):
        d[f"{nm}_1w"] = w1[col].reindex(items).fillna(0).to_numpy()
        d[f"{nm}_4w"] = w4[col].reindex(items).fillna(0).to_numpy()
    d["demand_1w"] = d["views_1w"] + d["atc_1w"]
    d["active"] = (d["views_1w"] + d["atc_1w"] + d["tx_1w"]) > 0
    # number of status changes in the last 4 snapshot steps
    lo = max(0, j - 4)
    seg = Sv[:, lo:j + 1]
    d["changes_4w"] = (np.abs(np.diff(seg, axis=1)) > 0).sum(axis=1) if seg.shape[1] > 1 else 0
    frames[j] = d

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 4 of 7)
# ---------- KPI 1 and KPI 3 (per snapshot) ----------
k1, k3 = [], []
for j in range(len(snaps)):
    d = frames[j]
    act = d[d["active"] & d["state"].notna()]
    oos = int((act["state"] == 0).sum())
    k1.append({"snapshot": dates[j], "active_items": len(act), "active_out_of_stock": oos,
               "oos_rate_pct": round(100 * oos / len(act), 1)})
    tot = d["demand_1w"].sum()
    un = d.loc[d["state"] == 0, "demand_1w"].sum()
    k3.append({"snapshot": dates[j], "views_plus_addtocart_7d": int(tot), "on_unavailable_items": int(un),
               "demand_on_unavailable_pct": round(100 * un / tot, 1)})
k1, k3 = pd.DataFrame(k1), pd.DataFrame(k3)

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 5 of 7)
# ---------- modelling table, KPI 2 (per week) and KPI 4 ----------
rows, k2w, k4w = [], [], []
cat_last = None
for j in range(len(snaps) - 1):
    if snaps[j + 1] - snaps[j] != 7 * DAY:      # skip weeks whose next snapshot is missing (14 Jun -> 28 Jun)
        continue
    d = frames[j]
    el = d[(d["state"] == 1) & d["active"]].copy()
    el["y_next_out"] = (Sv[:, j + 1][items.get_indexer(el.index)] == 0).astype(int)
    el["snapshot"] = dates[j]
    rows.append(el)
    # KPI 2: observed stock-out rate per category (stand-in for the model risk score)
    g = el.groupby("cat").agg(eligible_items=("y_next_out", "size"), observed_stockouts=("y_next_out", "sum"),
                              demand=("demand_1w", "sum"))
    g["wd"] = (el["demand_1w"] * el["y_next_out"]).groupby(el["cat"]).sum()
    g["observed_stockout_rate_pct"] = (100 * g["observed_stockouts"] / g["eligible_items"]).round(1)
    g["demand_weighted_risk"] = np.where(g["demand"] > 0, (100 * g["wd"] / g["demand"]).round(1), np.nan)
    g = g.drop(columns="wd")
    unk = g[g.index == UNKNOWN_CAT]               # "Unknown" is shown but never ranked: it is not a real category
    g = g[g.index != UNKNOWN_CAT]
    ok = g[g["eligible_items"] >= 20].sort_values("observed_stockout_rate_pct", ascending=False)
    n_el = len(ok); n_high = int(round(0.10 * n_el))
    k2w.append({"snapshot": dates[j], "next_snapshot": dates[j + 1], "eligible_categories": n_el, "high_risk_categories": n_high,
                "unknown_category_items": int(unk["eligible_items"].sum())})
    if j == len(snaps) - 2:                       # latest usable week: 6 Sep -> 13 Sep
        g["tier"] = "low data (<20 items)"
        ok = ok.copy()
        ok["tier"] = ["High"] * n_high + ["Medium"] * int(round(0.20 * n_el)) + ["Low"] * (n_el - n_high - int(round(0.20 * n_el)))
        unk = unk.assign(tier="Unknown category (not ranked)")
        cat_last = pd.concat([ok, g[g["eligible_items"] < 20].sort_values("eligible_items", ascending=False), unk]).reset_index()
        cat_last = cat_last.rename(columns={"cat": "categoryid"})
    # KPI 4 rule: predict stock-out when the item changed status in the last 4 weeks
    pred = (el["changes_4w"] > 0).astype(int)
    tp = int(((pred == 1) & (el["y_next_out"] == 1)).sum()); fp = int(((pred == 1) & (el["y_next_out"] == 0)).sum())
    fn = int(((pred == 0) & (el["y_next_out"] == 1)).sum()); tn = int(((pred == 0) & (el["y_next_out"] == 0)).sum())
    k4w.append({"snapshot": dates[j], "rows": len(el), "stockouts": int(el["y_next_out"].sum()), "TP": tp, "FP": fp, "FN": fn, "TN": tn})

mt = pd.concat(rows)
pooled = (len(items) * len(snaps), frames[0].shape[1])
log.add("stack item x snapshot tables", "item x snapshot", (len(items), frames[0].shape[1]), pooled,
        f"{len(items):,} items x {len(snaps)} snapshots")
n_weeks = len(rows)
log.add("drop weeks whose next snapshot is not 7 days later", "item x snapshot", pooled, (len(items) * n_weeks, pooled[1]),
        f"{len(snaps) - n_weeks} snapshots dropped (no label), {n_weeks} prediction weeks left")
log.add("keep in-stock and active items, add label y_next_out and snapshot", "item x snapshot -> modelling table",
        (len(items) * n_weeks, pooled[1]), (mt.shape[0], mt.shape[1] + 1),
        f"+1 column = itemid index; rows with Unknown category: {int(mt['cat_missing'].sum()):,}")
known = mt["cat_missing"] == 0
log.add('map category to parent category (unknown parent -> "Unknown")', "modelling table", (mt.shape[0], mt.shape[1] + 1), (mt.shape[0], mt.shape[1] + 1),
        f"{len(not_in_tree)} of {len(item_cats)} categoryid values in item_properties are not in category_tree "
        f"({int((known & (mt['cat_in_tree'] == 0)).sum()):,} modelling rows); "
        f"{int((known & (mt['parent_cat'] == mt['cat'])).sum()):,} modelling rows have a top-level category (own parent)")
mt.to_csv(WORK_DIR / "modelling_table.csv.gz", index_label="itemid")
# small table of all active items per snapshot (any status), used by step 4 for category demand
iw = pd.concat([frames[j].loc[frames[j]["active"], ["cat", "demand_1w"]].assign(snapshot=dates[j]) for j in range(len(snaps))])
iw.to_csv(WORK_DIR / "item_week.csv.gz", index_label="itemid")
k2w, k4 = pd.DataFrame(k2w), pd.DataFrame(k4w)

def split_of(d):
    return "train" if d <= "2015-07-26" else "validate" if d <= "2015-08-16" else "test"
k4["split"] = k4["snapshot"].map(split_of)
def metrics(df):
    s = df[["rows", "stockouts", "TP", "FP", "FN", "TN"]].sum()
    r = s.TP / (s.TP + s.FN); pr = s.TP / (s.TP + s.FP); f1 = 2 * r * pr / (r + pr)
    return pd.Series({**s.astype(int).to_dict(), "stockout_share_pct": round(100 * s.stockouts / s.rows, 1),
                      "recall": round(r, 2), "precision": round(pr, 2), "f1": round(f1, 2)})
k4["recall"] = (k4.TP / (k4.TP + k4.FN)).round(2); k4["precision"] = (k4.TP / (k4.TP + k4.FP)).round(2)
k4["f1"] = (2 * k4.recall * k4.precision / (k4.recall + k4.precision)).round(2)
k4_split = k4.groupby("split").apply(metrics, include_groups=False).reindex(["train", "validate", "test"])
k4_split.loc["ALL"] = metrics(k4)

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 6 of 7)
# ---------- KPI 3 by category (latest snapshot) and watch list ----------
dl = frames[len(snaps) - 1]
cat3 = dl.groupby("cat").agg(demand=("demand_1w", "sum")).join(
    dl[dl["state"] == 0].groupby("cat")["demand_1w"].sum().rename("demand_on_unavailable"))
cat3["demand_on_unavailable"] = cat3["demand_on_unavailable"].fillna(0)
cat3 = cat3[cat3["demand"] > 0]
cat3["demand_on_unavailable_pct"] = (100 * cat3["demand_on_unavailable"] / cat3["demand"]).round(1)
cat3 = cat3.sort_values("demand", ascending=False).reset_index().rename(columns={"cat": "categoryid"})
wl = dl[(dl["state"] == 1) & dl["active"] & (dl["changes_4w"] > 0)].sort_values("demand_1w", ascending=False).head(50)
wl = wl.reset_index()[["itemid", "cat", "views_1w", "atc_1w", "changes_4w"]].rename(
    columns={"cat": "categoryid", "views_1w": "views_7d", "atc_1w": "addtocart_7d", "changes_4w": "status_changes_4wk"})
wl["predicted_p_stockout"] = "to be added (model)"

In [ ]:
#@title Step 3: weekly item table, target, category fixes, KPI tables (part 7 of 7)
# ---------- reconciliation with the proposal ----------
last = len(snaps) - 1
k1d = k1.set_index("snapshot"); k3d = k3.set_index("snapshot")
n_hi = int(k2w.iloc[-1]["high_risk_categories"]); n_el = int(k2w.iloc[-1]["eligible_categories"])
tot_rows, tot_pos = int(mt.shape[0]), int(mt["y_next_out"].sum())
t = k4_split.loc["test"]
rec = pd.DataFrame([
    ("KPI1 OOS rate on 13 Sep (%)", 58, k1d.loc["2015-09-13", "oos_rate_pct"], 0.5),
    ("KPI1 OOS rate on 6 Sep (%)", 59, k1d.loc["2015-09-06", "oos_rate_pct"], 0.5),
    ("KPI1 min over 18 dates (%)", 43, k1["oos_rate_pct"].min(), 0.5),
    ("KPI1 max over 18 dates (%)", 60, k1["oos_rate_pct"].max(), 0.5),
    ("KPI2 eligible categories", 220, n_el, 0),
    ("KPI2 high-risk categories", 22, n_hi, 0),
    ("KPI3 demand on unavailable 13 Sep (%)", 43, k3d.loc["2015-09-13", "demand_on_unavailable_pct"], 0.5),
    ("KPI3 min (%)", 29, k3["demand_on_unavailable_pct"].min(), 0.5),
    ("KPI3 max (%)", 49, k3["demand_on_unavailable_pct"].max(), 0.5),
    ("Modelling rows", 283849, tot_rows, 0),
    ("Stock-outs in modelling table", 29502, tot_pos, 0),
    ("Stock-out share overall (%)", 10.4, round(100 * tot_pos / tot_rows, 1), 0.05),
    ("KPI4 rule recall (test)", 0.42, t["recall"], 0.005),
    ("KPI4 rule precision (test)", 0.24, t["precision"], 0.005),
    ("KPI4 rule F1 (test)", 0.30, t["f1"], 0.005),
    ("Stock-out share in test rows (%)", 11.3, t["stockout_share_pct"], 0.05),
], columns=["check", "proposal_says", "script_result", "tolerance"])
# tolerance = half of the last digit written in the proposal (58 -> 0.5, 10.4 -> 0.05, 0.42 -> 0.005); counts must be equal
rec["match"] = np.where((rec["proposal_says"].astype(float) - rec["script_result"].astype(float)).abs() <= rec["tolerance"] + 1e-9, "OK", "DIFF")

def show_unknown(df):
    """Write the -1 code as the word Unknown in the sheets."""
    df = df.copy()
    df["categoryid"] = df["categoryid"].astype("int64").astype(object).where(df["categoryid"] != UNKNOWN_CAT, "Unknown")
    return df
cat_last, cat3, wl = show_unknown(cat_last), show_unknown(cat3), show_unknown(wl)

with pd.ExcelWriter(OUT_DIR / "G31-04 KPI Tables.xlsx") as xw:
    pd.DataFrame({"sheet": ["KPI1_OOS_rate", "KPI2_by_week", "KPI2_categories_latest", "KPI3_demand_unavail", "KPI3_by_category",
                            "KPI4_rule_by_week", "KPI4_rule_by_split", "Watchlist", "Reconcile",
                            "Data_dictionary", "Cleaning_log"],
                  "what": ["Out-of-stock rate of active items, each snapshot", "High-risk category count per prediction week (observed rate)",
                           "Categories for 6 to 13 Sep: observed stock-out rate, demand-weighted risk, tier",
                           "Share of views+add-to-carts that go to unavailable items, each snapshot",
                           "Same share by category, snapshot 13 Sep", "Simple rule baseline: TP/FP/FN/TN, recall, precision, F1 per week",
                           "Same, pooled per split (train/validate/test)", "Top 50 in-stock items with a recent status change by demand, 13 Sep",
                           "Script result vs the numbers written in the proposal",
                           "Columns we use: type, meaning, example, why (written by step 5)",
                           "Rows and columns before and after each cleaning step (written by step 5)"]}).to_excel(xw, sheet_name="README", index=False)
    k1.to_excel(xw, sheet_name="KPI1_OOS_rate", index=False); k2w.to_excel(xw, sheet_name="KPI2_by_week", index=False)
    cat_last.to_excel(xw, sheet_name="KPI2_categories_latest", index=False); k3.to_excel(xw, sheet_name="KPI3_demand_unavail", index=False)
    cat3.to_excel(xw, sheet_name="KPI3_by_category", index=False); k4.to_excel(xw, sheet_name="KPI4_rule_by_week", index=False)
    k4_split.reset_index().to_excel(xw, sheet_name="KPI4_rule_by_split", index=False); wl.to_excel(xw, sheet_name="Watchlist", index=False)
    rec.to_excel(xw, sheet_name="Reconcile", index=False)
print(rec.to_string(index=False))
_free(_before)

### Step 4: demand features
Adds the demand features to the modelling table. Only data at or before the snapshot is used.

```text
Step 4: add demand features to the modelling table.

Input : work/modelling_table.csv.gz and work/item_week.csv.gz (step 3), work/item_props_small.csv.gz (step 1, for the
        exact snapshot timestamps), work/events_clean.csv.gz (step 1, for visitors and the time of the last purchase).
Output: work/modelling_table_features.csv.gz (same rows as the modelling table, more columns).

Every feature uses only events at or before the snapshot time t, so nothing from the label week leaks in.
Windows: 1w = (t - 7 days, t], 4w = (t - 28 days, t]. The reason for each feature is in docs/data_dictionary.md.
```
*Code copied from `scripts/04_demand_features.py`.*

In [ ]:
#@title Step 4: demand features (part 1 of 5)
_before = set(globals())
import numpy as np
import pandas as pd

DAY = 86_400_000
log = CleanLog("04_demand_features")
mt = pd.read_csv(WORK_DIR / "modelling_table.csv.gz")
log.add("load modelling table", "modelling_table", (0, 0), mt.shape)
shape_in = mt.shape

# exact snapshot timestamp for each snapshot date
p = pd.read_csv(WORK_DIR / "item_props_small.csv.gz", usecols=["timestamp", "property"])
snaps = np.sort(p.loc[p["property"] == "available", "timestamp"].unique())
t_of = dict(zip(pd.to_datetime(snaps, unit="ms").strftime("%Y-%m-%d"), snaps))
del p
ev = pd.read_csv(WORK_DIR / "events_clean.csv.gz", usecols=["timestamp", "visitorid", "event", "itemid"])
log.add("load cleaned events (4 of 5 columns)", "events_clean", (0, 0), ev.shape, "transactionid not needed")

In [ ]:
#@title Step 4: demand features (part 2 of 5)
# ---------- features from columns that step 3 already made ----------
mt["demand_4w"] = mt["views_4w"] + mt["atc_4w"]
mt["demand_prev3w_avg"] = (mt["demand_4w"] - mt["demand_1w"]) / 3
mt["demand_trend"] = (mt["demand_1w"] + 1) / (mt["demand_prev3w_avg"] + 1)
mt["atc_rate_1w"] = np.where(mt["views_1w"] > 0, mt["atc_1w"] / mt["views_1w"].where(mt["views_1w"] > 0), 0.0)
mt["buy_rate_4w"] = np.where(mt["views_4w"] > 0, mt["tx_4w"] / mt["views_4w"].where(mt["views_4w"] > 0), 0.0)

In [ ]:
#@title Step 4: demand features (part 3 of 5)
# ---------- features that need the events again ----------
vis, last_tx = [], []
weeks = set(mt["snapshot"])
for d, t in t_of.items():
    if d not in weeks:
        continue
    w = ev[(ev["timestamp"] > t - 7 * DAY) & (ev["timestamp"] <= t) & ev["event"].isin(["view", "addtocart"])]
    vis.append(w.groupby("itemid")["visitorid"].nunique().rename("visitors_1w").reset_index().assign(snapshot=d))
    tx = ev[(ev["timestamp"] <= t) & (ev["event"] == "transaction")]
    lt = tx.groupby("itemid")["timestamp"].max()
    last_tx.append(((t - lt) / DAY).round(2).rename("days_since_last_tx").reset_index().assign(snapshot=d))
mt = mt.merge(pd.concat(vis), on=["itemid", "snapshot"], how="left")
mt["visitors_1w"] = mt["visitors_1w"].fillna(0).astype(int)
mt = mt.merge(pd.concat(last_tx), on=["itemid", "snapshot"], how="left")   # empty = no purchase seen before t

In [ ]:
#@title Step 4: demand features (part 4 of 5)
# ---------- category demand (all active items of the category, in stock or not) ----------
iw = pd.read_csv(WORK_DIR / "item_week.csv.gz")
cd = iw[iw["cat"] != UNKNOWN_CAT].groupby(["snapshot", "cat"])["demand_1w"].sum().rename("cat_demand_1w").reset_index()
mt = mt.merge(cd, on=["snapshot", "cat"], how="left")                       # empty for Unknown category
mt["item_share_of_cat_demand"] = np.where(mt["cat_demand_1w"] > 0, mt["demand_1w"] / mt["cat_demand_1w"], np.nan)

In [ ]:
#@title Step 4: demand features (part 5 of 5)
# ---------- checks ----------
assert len(mt) == shape_in[0], "row count changed"
assert not mt.duplicated(["itemid", "snapshot"]).any(), "duplicate item-week rows"
new = [c for c in mt.columns if c not in pd.read_csv(WORK_DIR / "modelling_table.csv.gz", nrows=0).columns]
log.add("add demand features", "modelling_table -> modelling_table_features", shape_in, mt.shape,
        f"{len(new)} new columns: " + ", ".join(new))
mt.to_csv(WORK_DIR / "modelling_table_features.csv.gz", index=False)
print("\nFeature summary (all rows):")
print(mt[new].describe().T[["count", "mean", "50%", "max"]].round(3).to_string())
print("\nRows with empty days_since_last_tx:", int(mt["days_since_last_tx"].isna().sum()),
      "| rows with empty cat_demand_1w (Unknown category):", int(mt["cat_demand_1w"].isna().sum()))
_free(_before)

### Step 5: data dictionary and cleaning log
Writes the data dictionary (with real example values) and the cleaning log.

```text
Step 5: write the data dictionary and the cleaning log.

Outputs: docs/data_dictionary.md, docs/cleaning_log.md, and the sheets Data_dictionary and Cleaning_log
in outputs/G31-04 KPI Tables.xlsx (other sheets, including Reconcile, are not touched).
Example values are read from the data, never typed by hand. Only small parts of the raw CSVs are read.
```
*Code copied from `scripts/05_docs.py`.*

In [ ]:
#@title Step 5: data dictionary and cleaning log (part 1 of 4)
_before = set(globals())
import pandas as pd

DOCS = DOCS_DIR
XLSX = OUT_DIR / "G31-04 KPI Tables.xlsx"

In [ ]:
#@title Step 5: data dictionary and cleaning log (part 2 of 4)
# ---------- one real example row per table ----------
small = pd.read_csv(WORK_DIR / "item_props_small.csv.gz", nrows=200_000, dtype={"value": "string"})
ip_row = small[small["property"] == "available"].iloc[0]
ev = pd.read_csv(DATA_DIR / "events.csv", nrows=200_000)   # raw file: the dictionary describes the raw columns
ev_row = ev.iloc[0]
ex = {"item_properties (part1 + part2)": {c: str(ip_row[c]) for c in ip_row.index},
      "events": {c: str(ev_row[c]) for c in ("timestamp", "visitorid", "event", "itemid")}}
tid = ev["transactionid"].dropna()
ex["events"]["transactionid"] = "" if tid.empty else str(int(tid.iloc[0]))  # first non-empty one (purchases only)
ct = pd.read_csv(DATA_DIR / "category_tree.csv")
ct_row = ct.dropna().iloc[0]
ex["category_tree"] = {"categoryid": str(int(ct_row["categoryid"])), "parentid": str(int(ct_row["parentid"]))}
mt = pd.read_csv(WORK_DIR / "modelling_table_features.csv.gz")
# from 7 Jun on, the 28-day window lies fully inside the data (starts 3 May), so 4w features are not partial
pick = mt[(mt["snapshot"] >= "2015-06-07") & (mt["tx_4w"] > 0) & (mt["cat_missing"] == 0) & mt["days_since_last_tx"].notna()]
mt_row = (pick if len(pick) else mt).iloc[0]
mt_key = [k for k in SPEC if k.startswith("modelling table")][0]
ex[mt_key] = {c: (f"{v:.3f}".rstrip("0").rstrip(".") if isinstance(v, float) else str(v)) for c, v in mt_row.items()}

# check that the spec covers exactly the columns of the modelling table
spec_cols = [c[0] for c in SPEC[mt_key]]
all_cols = list(pd.read_csv(WORK_DIR / "modelling_table_features.csv.gz", nrows=0).columns)
missing, extra = set(all_cols) - set(spec_cols), set(spec_cols) - set(all_cols)
if missing or extra:
    raise SystemExit(f"dictionary_spec.py is out of date. Not described: {sorted(missing)}. Not in the table: {sorted(extra)}")

rows = []
for table, cols in SPEC.items():
    for col, typ, meaning, used, why in cols:
        rows.append({"table": table, "column": col, "type": typ, "meaning": meaning,
                     "example": ex.get(table, {}).get(col, ""), "used": used, "why": why})
dd = pd.DataFrame(rows)

In [ ]:
#@title Step 5: data dictionary and cleaning log (part 3 of 4)
# ---------- markdown ----------
def md_table(df):
    out = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    for _, r in df.iterrows():
        out.append("| " + " | ".join(str(v).replace("|", "/") for v in r) + " |")
    return "\n".join(out)

lines = ["# Data dictionary", "",
         "Generated by `scripts/05_docs.py` from `scripts/dictionary_spec.py`. Do not edit by hand: change the spec and rerun.",
         "Example values come from the data (first suitable row). In the modelling table all examples come from the same row.",
         "Windows: 1w = (t - 7 days, t], 4w = (t - 28 days, t], t = snapshot time. All features use events at or before t only.", ""]
for table in SPEC:
    lines += [f"## {table}", "", md_table(dd[dd["table"] == table].drop(columns="table")), ""]
(DOCS / "data_dictionary.md").write_text("\n".join(lines))

cl = pd.read_csv(LOG_PATH)
cl_lines = ["# Cleaning log", "",
            "Generated by `scripts/05_docs.py` from `work/cleaning_log.csv` (written by steps 1, 3 and 4). Rows x columns before and after each step.",
            "A `0 x 0` before means the table is read from file in that step.", "",
            md_table(cl.fillna("")), ""]
(DOCS / "cleaning_log.md").write_text("\n".join(cl_lines))

In [ ]:
#@title Step 5: data dictionary and cleaning log (part 4 of 4)
# ---------- workbook sheets ----------
with pd.ExcelWriter(XLSX, mode="a", engine="openpyxl", if_sheet_exists="replace") as xw:
    dd.to_excel(xw, sheet_name="Data_dictionary", index=False)
    cl.to_excel(xw, sheet_name="Cleaning_log", index=False)
print("wrote docs/data_dictionary.md, docs/cleaning_log.md and 2 sheets in", XLSX)
print(cl[["script", "step", "action", "rows_before", "cols_before", "rows_after", "cols_after"]].to_string(index=False))
_free(_before)

### Step 6: cleaned dataset (CSV) and comparison with the Progress 1 slides
Writes the cleaned modelling table as CSV for the submission, and puts our counts next to the numbers on the Progress 1 slides.

```text
Step 6: export the cleaned modelling table as CSV for the course submission, and compare our counts with
the numbers on the group's Progress 1 slides (12 Nov 2026).

Outputs: outputs/G31-04_Progress1_CleanedData.csv (not in git, too big), outputs/compare_progress1.csv
The slide numbers below are copied from the slide PDF, not computed. Our numbers are read from the cleaning log and the
KPI workbook, so they always come from the last run.
```
*Code copied from `scripts/06_export_compare.py`.*

In [ ]:
#@title Step 6: cleaned dataset (CSV) and comparison with the Progress 1 slides
_before = set(globals())
import pandas as pd

mt = pd.read_csv(WORK_DIR / "modelling_table_features.csv.gz")
out = OUT_DIR / "G31-04_Progress1_CleanedData.csv"
mt.to_csv(out, index=False)
print(f"wrote {out} ({len(mt):,} rows x {mt.shape[1]} columns, {out.stat().st_size / 1e6:.1f} MB)")

log = pd.read_csv(LOG_PATH)
def after(action_start):
    r = log[log["action"].str.startswith(action_start)]
    return int(r.iloc[-1]["rows_after"]) if len(r) else None

x = pd.read_excel(OUT_DIR / "G31-04 KPI Tables.xlsx", sheet_name=None)
k2 = x["KPI2_by_week"].iloc[-1]
k3 = x["KPI3_demand_unavail"].set_index("snapshot")
k4 = x["KPI4_rule_by_split"].set_index("split")
ours = {
    "Item property rows (raw)": after("stack part 1 and part 2"),
    "Rows with available or categoryid": after("keep property in"),
    "Status rows, first record and changes only": after("count first record"),
    "Item-weeks with a known status": after("item-weeks with a known status"),
    "Events after cleaning": after("drop exact duplicate events"),
    "Modelling table rows": len(mt),
    "Stock-outs in modelling table": int(mt["y_next_out"].sum()),
    "Eligible categories, week 6 to 13 Sep": int(k2["eligible_categories"]),
    "High-risk categories, week 6 to 13 Sep": int(k2["high_risk_categories"]),
    "KPI3 demand on unavailable, 13 Sep (%)": float(k3.loc["2015-09-13", "demand_on_unavailable_pct"]),
    "Rule recall, test weeks (%)": round(100 * float(k4.loc["test", "recall"]), 1),
    "Rule precision, test weeks (%)": round(100 * float(k4.loc["test", "precision"]), 1),
}
slide = {   # Progress 1 slides, G31-04_Progress_1_slide_presentation_.pdf
    "Item property rows (raw)": 20275902, "Rows with available or categoryid": 2291853,
    "Status rows, first record and changes only": 588801, "Item-weeks with a known status": 6707888,
    "Events after cleaning": 2755641, "Modelling table rows": 282708, "Stock-outs in modelling table": 29214,
    "Eligible categories, week 6 to 13 Sep": 225, "High-risk categories, week 6 to 13 Sep": 22,
    "KPI3 demand on unavailable, 13 Sep (%)": 43.6, "Rule recall, test weeks (%)": 47.6, "Rule precision, test weeks (%)": 22.0,
}
cmp = pd.DataFrame({"check": list(ours), "ours": list(ours.values()), "progress1_slide": [slide[k] for k in ours]})
cmp["match"] = ["same" if o == s else "DIFF" for o, s in zip(cmp["ours"], cmp["progress1_slide"])]
cmp.to_csv(OUT_DIR / "compare_progress1.csv", index=False)
print(cmp.to_string(index=False))
_free(_before)

### Step 7: all charts and pictures of the data
Every chart for the slides and the report, and a picture of each raw file and the modelling table (header and first real rows).

```text
Step 7: make every chart the group needs for slides and the report, from the outputs of steps 1 to 6.

Output: outputs/charts/*.png (1600 x 900 px, slide size). Every number in a chart is read from the run's own files:
work/cleaning_log.csv, work/modelling_table_features.csv.gz and outputs/G31-04 KPI Tables.xlsx. Nothing is typed by hand.

Charts
  01_cleaning_funnel.png      rows after each main cleaning step (explains 20M -> 2.3M -> ... -> modelling table)
  02_kpi_trend.png            KPI 1 (out-of-stock rate) and KPI 3 (demand on unavailable items) per weekly snapshot
  03_top_categories.png       categories with the highest observed stock-out rate, latest prediction week (KPI 2)
  04_target_balance.png       stock-out share per prediction week, with the train / validate / test split
  05_views_distribution.png   distribution of views in the last 7 days, and the stock-out rate per group
  06_feature_correlation.png  Spearman correlation of each feature with the target, train weeks only
  07_missing_values.png       share of empty values per column of the modelling table
  08_raw_item_properties.png  structure of the raw item_properties file: header and the first rows, plus rows of the 2 kept properties
  09_raw_events.png           structure of the raw events file: header and the first rows
  10_raw_category_tree.png    structure of the raw category tree: header and the first rows
  11_modelling_table.png      structure of the final modelling table: header and the first rows (first 9 columns)
The sample rows in charts 08 to 11 are read from the files themselves (the first rows), so they are real rows, not examples.
Colors: validated categorical slots (blue #2a78d6, orange #eb6834) and the blue/red diverging pair; light surface.
```
*Code copied from `scripts/07_charts.py`.*

In [ ]:
#@title Step 7: all charts and pictures of the data (part 1 of 9)
_before = set(globals())
import re
import textwrap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CH = OUT_DIR / "charts"
CH.mkdir(parents=True, exist_ok=True)
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
BLUE, ORANGE, RED, BLUE_LIGHT, BAND = "#2a78d6", "#eb6834", "#e34948", "#86b6ef", "#f0efec"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "font.size": 12, "axes.titlesize": 17, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False,
})
SPLIT = lambda d: "train" if d <= "2015-07-26" else "validate" if d <= "2015-08-16" else "test"
made = []


def heading(fig, title, subtitle):
    """Title and subtitle anchored to the figure, so a wide label column never pushes them off the page."""
    fig.text(0.02, 0.955, title, fontsize=16, fontweight="bold", va="top")
    fig.text(0.02, 0.885, textwrap.fill(subtitle, 115), fontsize=9.5, color=INK2, va="top", linespacing=1.3)


def new(title, subtitle, left=0.1):
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=200)
    fig.subplots_adjust(left=left, right=0.95, top=0.76, bottom=0.17)
    heading(fig, title, subtitle)
    return fig, ax


def save(fig, name, source):
    fig.text(0.01, 0.01, f"Source: {source}. Group G31-04, Dataset S1.", color=INK2, fontsize=8)
    fig.savefig(CH / name)
    plt.close(fig)
    made.append(name)


fmt = lambda v: f"{v:,.0f}"
log = pd.read_csv(LOG_PATH)
mt = pd.read_csv(WORK_DIR / "modelling_table_features.csv.gz")
x = pd.read_excel(OUT_DIR / "G31-04 KPI Tables.xlsx", sheet_name=None)


def rows_after(action_start):
    r = log[log["action"].str.startswith(action_start)]
    return int(r.iloc[-1]["rows_after"]) if len(r) else None

In [ ]:
#@title Step 7: all charts and pictures of the data (part 2 of 9)
# ---------- 01 cleaning funnel ----------
note = log.loc[log["action"].str.startswith("stack part 1 and part 2"), "note"]
m = re.search(r"([\d,]+) different properties", str(note.iloc[-1])) if len(note) else None
steps = [("Raw item properties" + (f"\n(all {m.group(1)} properties)" if m else ""), rows_after("stack part 1 and part 2")),
         ("Keep available\nand categoryid", rows_after("keep property in")),
         ("Status: first record\nand changes only", rows_after("count first record")),
         ("One row per item\nper week (forward fill)", rows_after("item-weeks with a known status")),
         ("Modelling table: in stock,\nactive, with a label", len(mt))]
steps = [(n, v) for n, v in steps if v is not None]
fig, ax = new(f"From {steps[0][1]:,} raw rows to the modelling table",
              "Rows after each main step of the item data. The 4th bar is larger than the 3rd because the format changes "
              "to one row per item and week (the last known status is carried forward), not because data is added.", left=0.27)
labels, vals = zip(*steps)
y = np.arange(len(vals))[::-1]
ax.barh(y, vals, color=BLUE, height=0.55)
for yi, v in zip(y, vals):
    ax.text(v + max(vals) * 0.01, yi, fmt(v), va="center", fontsize=11)
ax.set_yticks(y, labels, fontsize=10)
ax.set_xlim(0, max(vals) * 1.15)
big = max(vals) >= 2e6
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v / 1e6:.0f}M" if big else f"{v / 1e3:.0f}K"))
ax.grid(axis="y", visible=False)
save(fig, "01_cleaning_funnel.png", "work/cleaning_log.csv")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 3 of 9)
# ---------- 02 KPI trend ----------
k1, k3 = x["KPI1_OOS_rate"], x["KPI3_demand_unavail"]
d = pd.to_datetime(k1["snapshot"])
a1_, a2_ = k1["oos_rate_pct"], k3["demand_on_unavailable_pct"]
fig, ax = new(f"KPI 1 and KPI 3 per week: {a1_.iloc[0]:.1f}% to {a1_.iloc[-1]:.1f}% and {a2_.iloc[0]:.1f}% to {a2_.iloc[-1]:.1f}%",
              f"Each weekly snapshot, {k1['snapshot'].iloc[0]} to {k1['snapshot'].iloc[-1]} ({len(k1)} snapshots). Both values are percentages.")
for vals, col, name in [(k1["oos_rate_pct"], BLUE, "KPI 1: active items out of stock"),
                        (k3["demand_on_unavailable_pct"], ORANGE, "KPI 3: views and add-to-carts on unavailable items")]:
    ax.plot(d, vals, color=col, linewidth=2, marker="o", markersize=4, label=name)
    ax.annotate(f"{vals.iloc[-1]:.1f}%", (d.iloc[-1], vals.iloc[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=10.5, color=INK)
ax.set_ylim(0, 75)
ax.set_ylabel("%")
ax.legend(loc="lower right", fontsize=10)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%d %b"))
save(fig, "02_kpi_trend.png", "outputs/G31-04 KPI Tables.xlsx, sheets KPI1_OOS_rate and KPI3_demand_unavail")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 4 of 9)
# ---------- 03 top categories ----------
c = x["KPI2_categories_latest"]
c = c[c["tier"].isin(["High", "Medium"])].head(15).iloc[::-1]
if len(c) == 0:
    print("skip 03_top_categories.png: no category has 20 or more eligible items")
else:
    wk = x["KPI2_by_week"].iloc[-1]
    fig, ax = new(f"High-risk categories: {int(wk['high_risk_categories'])} of {int(wk['eligible_categories'])}",
                  f"Observed stock-out rate, week {wk['snapshot']} to {wk['next_snapshot']}. Top 15 categories with at least 20 items.")
    cols = [BLUE if t == "High" else BLUE_LIGHT for t in c["tier"]]
    yy = np.arange(len(c))
    ax.barh(yy, c["observed_stockout_rate_pct"], color=cols, height=0.62)
    for yi, (v, n) in enumerate(zip(c["observed_stockout_rate_pct"], c["eligible_items"])):
        ax.text(v + 0.8, yi, f"{v:.1f}%  (n={n})", va="center", fontsize=9.5)
    ax.set_yticks(yy, [f"Cat {v}" for v in c["categoryid"]], fontsize=9.5)
    ax.set_xlim(0, c["observed_stockout_rate_pct"].max() * 1.25)
    ax.set_xlabel("% of eligible items out of stock the next week")
    ax.grid(axis="y", visible=False)
    handles = [matplotlib.patches.Patch(color=col, label=lab) for t, col, lab in
               [("High", BLUE, "High (top 10%)"), ("Medium", BLUE_LIGHT, "Medium (next 20%)")] if t in set(c["tier"])]
    ax.legend(handles=handles, loc="lower right", fontsize=10)
    save(fig, "03_top_categories.png", "outputs/G31-04 KPI Tables.xlsx, sheet KPI2_categories_latest")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 5 of 9)
# ---------- 04 target balance ----------
tb = mt.groupby("snapshot")["y_next_out"].agg(["mean", "size"]).reset_index()
tb["pct"] = 100 * tb["mean"]
overall = 100 * mt["y_next_out"].mean()
fig, ax = new(f"Target balance: {overall:.1f}% of rows run out of stock the next week",
              f"Stock-out share per prediction week ({len(tb)} weeks). A model that always says 'stays in stock' is {100 - overall:.0f}% right and useless.")
xx = np.arange(len(tb))
for split in ["train", "validate", "test"]:
    idx = [i for i, s in enumerate(tb["snapshot"]) if SPLIT(s) == split]
    ax.axvspan(idx[0] - 0.5, idx[-1] + 0.5, color=BAND if split != "validate" else SURFACE, zorder=0)
    ax.text((idx[0] + idx[-1]) / 2, 0.97, split, transform=ax.get_xaxis_transform(), ha="center", va="top",
            color=INK2, fontsize=10)
ax.bar(xx, tb["pct"], color=BLUE, width=0.62)
ax.axhline(overall, color=INK2, linewidth=1, linestyle="--", label=f"overall {overall:.1f}%")
ax.legend(loc="upper left", bbox_to_anchor=(0, 0.9), fontsize=9.5)
ax.set_ylim(0, tb["pct"].max() * 1.3)
for i in [int(tb["pct"].idxmax()), int(tb["pct"].idxmin())]:
    ax.text(i, tb["pct"][i] + 0.4, f"{tb['pct'][i]:.1f}%", ha="center", fontsize=9.5)
ax.set_xticks(xx, pd.to_datetime(tb["snapshot"]).dt.strftime("%d %b"), rotation=45, ha="right", fontsize=9)
ax.set_ylabel("% stock-outs")
ax.grid(axis="x", visible=False)
save(fig, "04_target_balance.png", "work/modelling_table_features.csv.gz")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 6 of 9)
# ---------- 05 views distribution ----------
bins = [-1, 0, 1, 2, 5, 10, 20, 50, np.inf]
names = ["0", "1", "2", "3-5", "6-10", "11-20", "21-50", "51+"]
g = mt.assign(b=pd.cut(mt["views_1w"], bins=bins, labels=names)).groupby("b", observed=False)["y_next_out"].agg(["size", "mean"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8, 4.5), dpi=200, sharey=True)
fig.subplots_adjust(left=0.1, right=0.95, top=0.7, bottom=0.12, wspace=0.08)
heading(fig, "Views in the last 7 days and the stock-out rate",
        f"All {len(mt):,} rows of the modelling table, grouped by views in the 7 days before the snapshot. "
        f"Median views: {mt['views_1w'].median():.0f}; max: {mt['views_1w'].max():,.0f}.")
yy = np.arange(len(g))[::-1]
share = 100 * g["size"] / g["size"].sum()
a1.barh(yy, share, color=BLUE, height=0.62)
a2.barh(yy, 100 * g["mean"], color=ORANGE, height=0.62)
for a, vals, t in [(a1, share, "Share of rows (%)"), (a2, 100 * g["mean"], "Stock-out rate next week (%)")]:
    for yi, v in zip(yy, vals):
        a.text(v + vals.max() * 0.02, yi, f"{v:.1f}", va="center", fontsize=9.5)
    a.set_title(t, fontsize=11.5, loc="left", pad=4)
    a.set_xlim(0, vals.max() * 1.25)
    a.grid(axis="y", visible=False)
a1.set_yticks(yy, names)
a1.set_ylabel("Views in last 7 days")
save(fig, "05_views_distribution.png", "work/modelling_table_features.csv.gz")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 7 of 9)
# ---------- 06 feature correlation ----------
skip = {"itemid", "snapshot", "y_next_out", "state", "active", "cat", "parent_cat"}
feats = [f for f in mt.columns if f not in skip and pd.api.types.is_numeric_dtype(mt[f])]
tr = mt[mt["snapshot"].map(SPLIT) == "train"]
def spearman(a, b):
    """Spearman correlation = Pearson correlation of the ranks (no scipy needed). Rows with an empty value are left out."""
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank())
corr = pd.Series({f: spearman(tr[f], tr["y_next_out"]) for f in feats}).dropna()
corr = corr.reindex(corr.abs().sort_values().index)
fig, ax = new(f"Correlation of each feature with the target (largest |r| = {corr.abs().max():.3f})",
              f"Spearman correlation with 'out of stock next week', train weeks only ({len(tr):,} rows). "
              "Blue = positive, red = negative. Codes (cat, parent_cat) are left out.", left=0.26)
yy = np.arange(len(corr))
ax.barh(yy, corr.values, color=[BLUE if v >= 0 else RED for v in corr.values], height=0.62)
for yi, v in zip(yy, corr.values):
    off = corr.abs().max() * 0.03
    ax.text(v + (off if v >= 0 else -off), yi, f"{v:+.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=8.5)
ax.set_yticks(yy, corr.index, fontsize=8.5)
ax.axvline(0, color=INK2, linewidth=0.8)
lim = corr.abs().max() * 1.35
ax.set_xlim(-lim, lim)
ax.grid(axis="y", visible=False)
save(fig, "06_feature_correlation.png", "work/modelling_table_features.csv.gz, train weeks 10 May to 26 Jul 2015")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 8 of 9)
# ---------- 07 missing values ----------
miss = (100 * mt.isna().mean()).sort_values()
miss = miss[miss > 0]
unknown = 100 * mt["cat_missing"].mean()
fig, ax = new("Missing values in the modelling table",
              f"Share of empty cells per column; columns with no empty cells are not shown. "
              f"Also: category 'Unknown' (code -1, not empty) in {unknown:.1f}% of rows.", left=0.28)
yy = np.arange(len(miss))
ax.barh(yy, miss.values, color=BLUE, height=0.5)
for yi, v in zip(yy, miss.values):
    ax.text(v + 1, yi, f"{v:.1f}%", va="center", fontsize=10.5)
ax.set_yticks(yy, miss.index)
ax.set_xlim(0, 100)
ax.set_xlabel("% of rows empty")
ax.grid(axis="y", visible=False)
save(fig, "07_missing_values.png", "work/modelling_table_features.csv.gz")

In [ ]:
#@title Step 7: all charts and pictures of the data (part 9 of 9)
# ---------- 08 to 11: what the data looks like (header + first rows) ----------
HEAD = "#5f6a86"


def table_panel(ax, df, top_title):
    """Draw a header row and the sample rows as a table on an empty axis."""
    ax.axis("off")
    ax.set_title(top_title, fontsize=11, loc="left", color=INK2, fontweight="normal", pad=4)
    cut = lambda t: t if len(t) <= 30 else t[:27] + "..."      # very long coded values are shortened for display only
    cells = [[cut("" if pd.isna(v) else str(v)) for v in row] for row in df.itertuples(index=False)]
    t = ax.table(cellText=cells, colLabels=list(df.columns), loc="upper left", cellLoc="left", colLoc="left",
                 bbox=[0, 0, 1, 1])
    t.auto_set_font_size(False)
    t.set_fontsize(9.5)
    width = [max([len(str(c))] + [len(row[i]) for row in cells]) + 2 for i, c in enumerate(df.columns)]
    for (r, c), cell in t.get_celld().items():
        cell.set_width(width[c] / sum(width))
        cell.set_edgecolor(GRID)
        if r == 0:
            cell.set_facecolor(HEAD)
            cell.get_text().set_color("white")
            cell.get_text().set_fontweight("bold")
        else:
            cell.set_facecolor("white" if r % 2 else BAND)
            cell.get_text().set_family("monospace")


def utc(ms):
    return pd.to_datetime(int(ms), unit="ms").strftime("%d %b %Y %H:%M:%S UTC")


def size_of(action, table):
    r = log[(log["action"] == action) & (log["table"] == table)]
    return (int(r.iloc[-1]["rows_after"]), int(r.iloc[-1]["cols_after"])) if len(r) else None


N = 5
# item properties: first raw rows, then rows of the 2 properties we keep
ip = pd.read_csv(DATA_DIR / "item_properties_part1.csv", nrows=N, dtype=str)
small = pd.read_csv(WORK_DIR / "item_props_small.csv.gz", nrows=50_000, dtype=str)
kept = pd.concat([small[small["property"] == "available"].head(2), small[small["property"] == "categoryid"].head(2)])
sz1, sz2 = size_of("read raw file", "item_properties_part1.csv"), size_of("read raw file", "item_properties_part2.csv")
fig = plt.figure(figsize=(8, 4.5), dpi=200)
heading(fig, "Raw file: item_properties_part1.csv (part 2 has the same 4 columns)",
        (f"Part 1: {sz1[0]:,} rows x {sz1[1]} columns" if sz1 else "") + (f"; part 2: {sz2[0]:,} rows. " if sz2 else ". ")
        + "Long format: 1 row = 1 property value of 1 item at 1 time.")
a1 = fig.add_axes([0.03, 0.42, 0.94, 0.3])
table_panel(a1, ip, f"First {N} rows (coded names and values). Row 1: {ip['timestamp'].iloc[0]} ms = {utc(ip['timestamp'].iloc[0])}")
a2 = fig.add_axes([0.03, 0.08, 0.94, 0.24]); table_panel(a2, kept, "Rows of the 2 properties we keep (step 1 output): available (0/1) and categoryid")
save(fig, "08_raw_item_properties.png", "first rows of data_raw/item_properties_part1.csv and work/item_props_small.csv.gz")

ev = pd.read_csv(DATA_DIR / "events.csv", nrows=N, dtype=str)
ev_tx = pd.read_csv(DATA_DIR / "events.csv", nrows=200_000, dtype=str)
ev_tx = ev_tx[ev_tx["event"] == "transaction"].head(1)
sz = size_of("read raw file", "events.csv")
fig = plt.figure(figsize=(8, 4.5), dpi=200)
heading(fig, "Raw file: events.csv",
        (f"{sz[0]:,} rows x {sz[1]} columns. " if sz else "") + "1 row = 1 customer action (view, addtocart or transaction) on 1 item. "
        "transactionid is empty unless the event is a transaction.")
a1 = fig.add_axes([0.03, 0.40, 0.94, 0.33])
table_panel(a1, ev, f"First {N} rows of the file. Row 1 time: {ev['timestamp'].iloc[0]} ms = {utc(ev['timestamp'].iloc[0])}")
if len(ev_tx):
    a2 = fig.add_axes([0.03, 0.12, 0.94, 0.14]); table_panel(a2, ev_tx, "First transaction row in the file (has a transactionid)")
save(fig, "09_raw_events.png", "first rows of data_raw/events.csv")

ct = pd.read_csv(DATA_DIR / "category_tree.csv", nrows=N + 2, dtype=str)
sz = size_of("load category tree", "category_tree")
fig = plt.figure(figsize=(8, 4.5), dpi=200)
heading(fig, "Raw file: category_tree.csv",
        (f"{sz[0]:,} rows x {sz[1]} columns. " if sz else "") + "1 row = 1 category and its parent category. "
        "parentid is empty for a top-level category.")
a1 = fig.add_axes([0.25, 0.12, 0.5, 0.6]); table_panel(a1, ct, f"First {len(ct)} rows of the file (an empty parentid = top-level category)")
save(fig, "10_raw_category_tree.png", "first rows of data_raw/category_tree.csv")

cols9 = ["itemid", "snapshot", "cat", "parent_cat", "views_1w", "atc_1w", "tx_1w", "changes_4w", "y_next_out"]
sample = mt[[c for c in cols9 if c in mt.columns]].head(N).copy()
for c in sample.columns:   # show whole numbers without ".0"
    if pd.api.types.is_float_dtype(sample[c]) and (sample[c].dropna() % 1 == 0).all():
        sample[c] = sample[c].astype("Int64")
fig = plt.figure(figsize=(8, 4.5), dpi=200)
heading(fig, "Final table: modelling table (input for the models)",
        f"{len(mt):,} rows x {mt.shape[1]} columns. 1 row = 1 in-stock, active item in 1 week. "
        f"Target y_next_out = 1 if the item is out of stock 7 days later. Showing {len(sample.columns)} of {mt.shape[1]} columns; "
        "all columns are described in docs/data_dictionary.md.")
a1 = fig.add_axes([0.03, 0.2, 0.94, 0.5]); table_panel(a1, sample, f"First {N} rows")
save(fig, "11_modelling_table.png", "work/modelling_table_features.csv.gz")

print(f"{len(made)} charts in {CH}:")
for m in made:
    print("  ", m)
_free(_before)

## 3. Results

In [ ]:
#@title Cleaning log: rows and columns before and after each step
import pandas as pd
pd.set_option("display.max_colwidth", 140)
pd.read_csv(LOG_PATH)[["script", "step", "action", "rows_before", "cols_before", "rows_after", "cols_after", "note"]]

In [ ]:
#@title Reconcile: our results vs the numbers written in the proposal (OK = same within the proposal's rounding)
pd.read_excel(OUT_DIR / "G31-04 KPI Tables.xlsx", sheet_name="Reconcile")

In [ ]:
#@title Our counts vs the numbers on the Progress 1 slides
pd.read_csv(OUT_DIR / "compare_progress1.csv")

In [ ]:
#@title Modelling table: size, target balance, first rows, summary of every column
mt = pd.read_csv(WORK_DIR / "modelling_table_features.csv.gz")
print(f"{len(mt):,} rows x {mt.shape[1]} columns | stock-outs next week: {int(mt.y_next_out.sum()):,} ({100 * mt.y_next_out.mean():.1f}%)")
display(mt.head())
mt.describe().T[["count", "mean", "50%", "max"]].round(3)

In [ ]:
#@title Data dictionary (also in G31-04_run/docs/data_dictionary.md and in the Excel file)
pd.read_excel(OUT_DIR / "G31-04 KPI Tables.xlsx", sheet_name="Data_dictionary")

In [ ]:
#@title All charts (files in G31-04_run/outputs/charts)
from IPython.display import Image
for f in sorted(glob.glob(str(OUT_DIR / "charts" / "*.png"))):
    print(Path(f).name)
    display(Image(f, width=1000))

In [ ]:
#@title Data flow diagram (drawn by the group; picture stored inside this notebook)
import base64
from IPython.display import Image
display(Image(data=base64.b64decode("..."), width=1100))

In [ ]:
#@title Save everything as one zip (Colab downloads it to your computer)
zip_path = RUN_DIR / "G31-04_Progress1_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (OUT_DIR, DOCS_DIR):
        for f in sorted(folder.rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(RUN_DIR))
    z.write(LOG_PATH, LOG_PATH.relative_to(RUN_DIR))
names = zipfile.ZipFile(zip_path).namelist()
print(f"{zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB, {len(names)} files)")
for n in names:
    print("  ", n)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))